# Evaluate SAM OBSEA output

Grab grand truth annotations from OBSEA dataset. Rectify them to be parallel to the image axes. Compare to SAMv1 outputs. 


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
import os 
from PIL import Image
import cv2
import pylab
from sklearn import metrics
import glob, os

### Load datasets

In [ ]:
# All annotations
df = pd.read_csv('obsea/PANGAEA.946149.csv', sep='\t')

# get the list of images if already downloaded
imgs = glob.glob(os.path.join('obsea/image_data/', '*.jpg'))
imgs = [os.path.basename(item) for item in imgs]

# get annotations for those images
df = df[df['IMAGE'].isin(imgs[0:25])]

Define function to retrieve the parallel bboxes.

In [ ]:
def retrieve_parallel(orig):

    orig = np.asarray(orig)
    bx = cv2.minAreaRect(orig)  # computes center, width, height, and angle

    xx = int(bx[0][0] - (bx[1][0]/2))
    yy = int(bx[0][1] - (bx[1][1]/2))

    if xx < 0:
        xx = 0

    if yy < 0:
        yy = 0

    if bx[2] == 90 or bx[2] == 0:
        # if the box is already parallel to axis, then just use min x,y as upper left
        zz = np.asarray(orig).min(axis=0)

        if bx[2] == 0:
            return pd.Series([int(zz[0]), int(zz[1]), int(bx[1][0]), int(bx[1][1]), bx[2]])
        else: 
            return pd.Series([int(zz[0]), int(zz[1]), int(bx[1][1]), int(bx[1][0]), bx[2]])
    elif bx[2] > 45:
        xx = int(bx[0][0] - (bx[1][1]/2))
        yy = int(bx[0][1] - (bx[1][0]/2))
        if xx < 0:
            xx = 0
        if yy < 0:
            yy = 0
        return pd.Series([int(xx), int(yy), int(bx[1][1]), int(bx[1][0]), bx[2]])
    else:
        # return x, y, w, h
        xx = int(bx[0][0] - (bx[1][0]/2))
        yy = int(bx[0][1] - (bx[1][1]/2))
        if xx < 0:
            xx = 0
        if yy < 0:
            yy = 0
        return pd.Series([int(xx), int(yy), int(bx[1][0]), int(bx[1][1]), bx[2]])

    # return pd.Series([xx, yy, int(bx[1][0]), int(bx[1][1])])

df[['xx', 'yy', 'ww', 'hh', 'theta']] = df.apply(
    lambda x: retrieve_parallel([(x[f'bboxx{ii} [pixel]'], x[f'bboxy{ii} [pixel]']) for ii in range(1,5)]), axis=1
)

Load in the SAM annotations.

In [ ]:
sam = COCO('obsea/SAM-output-no-score.json')

Filter out any obviously wrong annotations. Shouldn't be any for OBSEA dataset.

In [ ]:
anns = sam.loadAnns(sam.getAnnIds(imgIds=sam.getImgIds()))
filt_anns = [item['area'] for item in anns if item['area'] < 1000000]

print(f"Number of annotations: {len(anns)}")
print(f"Filtered annotations: {len(filt_anns)}")

### Compute distance

Get the distance between the bounding box center and the nearest annotation point.

#### SAM v. points
SAM bboxes compared to original box annotations (after rotating to align with axis).

In [ ]:
out = []
for ii in sam.getImgIds():

    # get relevent annotated boxes and compute center coordinate
    tmp = df[df['IMAGE']==sam.imgs[ii]['file_name']].copy()

    gt = []
    for item in tmp.itertuples():
        center = [item.xx + item.ww/2, item.yy + item.hh/2]
        gt.append(center)

    # get the annotations
    anns = sam.loadAnns(sam.getAnnIds(imgIds=ii))

    # pull out coordinates of center of bounding box and area

    test = []
    for ann in anns:
        bbox = ann['bbox']
        center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
        test.append(center)

    # compute distance between every bbox center and human annotated point
    zz = metrics.pairwise.euclidean_distances(np.array(test), np.array(gt))

    out.extend(np.amin(zz, axis=1).tolist())  # add the those distances to the output list

out = np.asarray(out)

print(f'med dist: {np.median(out)}')
print(f'mean dist: {np.mean(out)}')
print(f'max dist: {np.max(out)}')

# make a plot
counts, bins = np.histogram(out)

fig, ax = plt.subplots()
_ = ax.hist(bins[:-1], bins, weights=counts)
ax.set_xlabel('Euclidean distance (pixels)', fontsize=16)
ax.set_ylabel('Counts', fontsize=16)
ax.tick_params(labelsize=14)
fig.tight_layout()

### Plot individual frame

Image plotting with both SAM and ground truth bboxes.

In [ ]:
ii = sam.getImgIds()[23]

image = cv2.imread(f"obsea/image_data/{sam.imgs[ii]['file_name']}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# compare all annotations 
tmp = df[df['IMAGE']==sam.imgs[ii]['file_name']].copy()

gt = []
for item in tmp.itertuples():
    center = [item.yy + item.hh/2, item.xx + item.ww/2]
    gt.append(center)

test = []
for ann in sam.loadAnns(sam.getAnnIds(imgIds=ii)):
    bbox = ann['bbox']
    center = [bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2]
    test.append(center)

# plot the center of SAM bounding boxes
for cen in test:
    cv2.circle(image, (int(cen[0]), int(cen[1])), 7, (255, 127, 14), cv2.FILLED)

# plot the center of ground truth bboces
for cen in gt:
    cv2.circle(image, (int(cen[1]), int(cen[0])), 7, (0, 255, 0), cv2.FILLED)

fig, ax = plt.subplots(figsize=(16,20))
ax.imshow(image); ax.axis('off')
annIds = sam.getAnnIds(imgIds=ii)
anns = sam.loadAnns(annIds)

# plot the rotated bounding boxes
polygons = []
for item in tmp.itertuples():
    [bbox_x, bbox_y, bbox_w, bbox_h] = [item.xx, item.yy, item.ww, item.hh]
    poly = [[bbox_x, bbox_y], [bbox_x, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y]]
    np_poly = np.array(poly).reshape((4,2))
    polygons.append(Polygon(np_poly))
    #color.append(c)

c = [0.0, 1.0, 0.0]
p = PatchCollection(polygons, facecolor='none', linewidths=0, alpha=0.4)
ax.add_collection(p)
p = PatchCollection(polygons, facecolor='none', edgecolors=c, linewidths=2)
ax.add_collection(p)

# plot the sam proposals
polygons = []
for ann in anns:
    [bbox_x, bbox_y, bbox_w, bbox_h] = ann['bbox']
    poly = [[bbox_x, bbox_y], [bbox_x, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y]]
    np_poly = np.array(poly).reshape((4,2))
    polygons.append(Polygon(np_poly))

c = [1.0, 0.4980392156862745, 0.054901960784313725]
p = PatchCollection(polygons, facecolor='none', linewidths=0, alpha=0.4)
ax.add_collection(p)
p = PatchCollection(polygons, facecolor='none', edgecolors=c, linewidths=2)
ax.add_collection(p)

### Plot individual frame with seg

Image plotting with original bboxes and segmentation masks

In [ ]:
ii = sam.getImgIds()[23]

image = cv2.imread(f"obsea/image_data/{sam.imgs[ii]['file_name']}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# plot the segmentation masks
for ann in sam.loadAnns(sam.getAnnIds(imgIds=ii)):
    poly = np.array(ann['segmentation'])
    cv2.drawContours(image, [poly], 0, (255, 127, 14), 2)

# compare all annotations 
tmp = df[df['IMAGE']==sam.imgs[ii]['file_name']].copy()

gt = []
for item in tmp.itertuples():
    center = [item.yy + item.hh/2, item.xx + item.ww/2]
    gt.append(center)

# plot the center of ground truth bboxes
for cen in gt:
    cv2.circle(image, (int(cen[1]), int(cen[0])), 7, (0, 255, 0), cv2.FILLED)

fig, ax = plt.subplots(figsize=(16,20))
ax.imshow(image); ax.axis('off')
annIds = sam.getAnnIds(imgIds=ii)
anns = sam.loadAnns(annIds)

# plot the rotated bounding boxes
polygons = []
for item in tmp.itertuples():
    [bbox_x, bbox_y, bbox_w, bbox_h] = [item.xx, item.yy, item.ww, item.hh]
    poly = [[bbox_x, bbox_y], [bbox_x, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y+bbox_h], [bbox_x+bbox_w, bbox_y]]
    np_poly = np.array(poly).reshape((4,2))
    polygons.append(Polygon(np_poly))

c = [0.0, 1.0, 0.0]
p = PatchCollection(polygons, facecolor='none', linewidths=0, alpha=0.4)
ax.add_collection(p)
p = PatchCollection(polygons, facecolor='none', edgecolors=c, linewidths=2)
ax.add_collection(p)

### Compute mean IoU 

Compute the mean intersection over union between SAM output and ground truth.

In [ ]:
gt = COCO('obsea/gt-rectified-coco.json')
sam =COCO('obsea/SAM-output-with-score.json')

# Create evaluation object
eval = COCOeval(gt, sam, 'bbox')
eval.evaluate()

# compute IoU just for benthocodon
out = []
for ii in gt.getImgIds():
    for jj in sam.getCatIds():
        test = eval.computeIoU(ii, jj)
        if len(test)>0:
            #out.extend(np.amax(test, axis=0))  # all gt anns (some zeros where sam failed/no original points)
            out.extend(np.amax(test, axis=1))  # sam anns (some zeros where there is original )

out = np.asarray(out)

print(f'med iou: {np.median(out)}')
print(f'mean iou: {np.mean(out)}')
print(f'max iou: {np.max(out)}')

# make a plot
counts, bins = np.histogram(out)

fig, ax = plt.subplots()
_ = ax.hist(bins[:-1], bins, weights=counts)
ax.set_xlabel('Intersection over Union', fontsize=16)
ax.set_ylabel('Counts', fontsize=16)
ax.tick_params(labelsize=14)
ax.set_xlim([0,1])
fig.tight_layout()